# Visual Diagnostics: Manual Trajectory Picker

Cherry-pick one trajectory per encounter bin for the official
`evaluation.animate_best --selection-file` videos.

Workflow:

1. Run the setup cells once to load EGNN, HGNN, the constant-velocity baseline,
   the stratified test set, and the three full rollouts.
2. Set `BIN_NAME` to the bin you want to inspect: `close`, `near`, `mid`, `wide`, or `far`.
   Set `RANDOM_SEED` to `None` for a fresh random sample on every rerun, or to an
   integer to get a reproducible pick.
3. Rerun the picker cell to draw a random trajectory index from that bin.
4. Rerun the preview cell to see four side-by-side panels: ground truth, EGNN, HGNN,
   constant velocity.
5. When you see one you like, copy its index into the `SELECTED` mapping. Repeat for
   each bin.
6. Run the final cell to validate and write the YAML to `SELECTION_PATH` for the
   `--selection-file` animation pipeline.

The notebook is intentionally manual: it helps surface candidates, but the keep / discard
decision is yours.


In [ ]:
import sys
from pathlib import Path


def _find_impl_root() -> Path:
    """Walk up from CWD until we find the impl/ root."""
    here = Path.cwd().resolve()
    for cand in [here, *here.parents]:
        if (cand / "pyproject.toml").exists() and (cand / "evaluation").is_dir():
            return cand
    msg = f"could not locate impl/ root from {here}"
    raise RuntimeError(msg)


IMPL_ROOT = _find_impl_root()
if str(IMPL_ROOT) not in sys.path:
    sys.path.insert(0, str(IMPL_ROOT))
print(f"impl root: {IMPL_ROOT}")

In [ ]:
import numpy as np
import torch
import yaml
from matplotlib import pyplot as plt

from data._io import read_trajectories
from data._types import Trajectories
from evaluation._loader import load_trained_model
from evaluation.metrics import run_all_rollouts
from models.baselines import ConstantVelocityBaseline

## User parameters

Set the paths, the bin to sample from, and the random seed. Rerun the picker cell
without restarting the kernel to sample new candidates.

In [ ]:
EGNN_CHECKPOINT = IMPL_ROOT / "runs/egnn/20260510_211102/best.pt"
HGNN_CHECKPOINT = IMPL_ROOT / "runs/hgnn/20260510_211151/best.pt"
EGNN_CONFIG = IMPL_ROOT / "configs/egnn.yaml"
HGNN_CONFIG = IMPL_ROOT / "configs/hgnn.yaml"
TEST_PATH = IMPL_ROOT / "data/output/test.h5"
SELECTION_PATH = IMPL_ROOT / "runs/reports/official_1k/selections/manual.yaml"

DEVICE = "cpu"

BIN_NAME = "mid"  # close | near | mid | wide | far
RANDOM_SEED = None  # None for a fresh sample on every rerun, int for reproducibility

## One-time setup

Load both trained models, build the constant-velocity baseline, read the
stratified test bundle, and run autoregressive rollouts for all three predictors.
Rerun this cell only when you change a checkpoint or test path.

In [ ]:
device = torch.device(DEVICE)

egnn_bundle = load_trained_model(EGNN_CONFIG, EGNN_CHECKPOINT, device)
hgnn_bundle = load_trained_model(HGNN_CONFIG, HGNN_CHECKPOINT, device)
if egnn_bundle.cfg.data.dt != hgnn_bundle.cfg.data.dt:
    msg = (
        f"EGNN/HGNN configs disagree on dt: {egnn_bundle.cfg.data.dt} vs {hgnn_bundle.cfg.data.dt}"
    )
    raise ValueError(msg)
baseline_dt = float(egnn_bundle.cfg.data.dt)
baseline = ConstantVelocityBaseline(dt=baseline_dt).to(device).eval()
print(f"EGNN: epoch {egnn_bundle.checkpoint.epoch}, val_loss {egnn_bundle.checkpoint.val_loss:.6f}")
print(f"HGNN: epoch {hgnn_bundle.checkpoint.epoch}, val_loss {hgnn_bundle.checkpoint.val_loss:.6f}")
print(f"constant velocity baseline dt={baseline_dt}")

test_bundle = read_trajectories(TEST_PATH)
if test_bundle.encounter_bins is None:
    msg = f"test bundle at {TEST_PATH} is not stratified; encounter_bins is None"
    raise ValueError(msg)
print(
    f"test bundle: {test_bundle.states.shape}, bins: {[b.name for b in test_bundle.encounter_bins]}"
)

egnn_predicted = run_all_rollouts(egnn_bundle.model, test_bundle.states, device)
hgnn_predicted = run_all_rollouts(hgnn_bundle.model, test_bundle.states, device)
baseline_predicted = run_all_rollouts(baseline, test_bundle.states, device)
print(
    f"rollouts: egnn {egnn_predicted.shape} | hgnn {hgnn_predicted.shape} | baseline {baseline_predicted.shape}"
)

## Helpers

Defined inline so the notebook is self-contained. Three helpers:

- `random_trajectory_from_bin` draws one trajectory index uniformly at random from a
  named bin, with a strict guard against unknown / empty bins.
- `mean_rollout_position_mse` matches the criterion used by
  `evaluation.animate_best.select_best_trajectories` so panel titles surface the same
  number the automatic selector would use.
- `plot_trajectory_quad` lays out one row of four panels (truth, EGNN, HGNN, baseline)
  with shared axes.

In [ ]:
_PARTICLE_COLORS = ("tab:blue", "tab:red", "tab:green", "tab:purple", "tab:brown", "tab:cyan")


def _particle_color(index: int) -> str:
    """Stable color for one particle across every panel."""
    return _PARTICLE_COLORS[index % len(_PARTICLE_COLORS)]


def random_trajectory_from_bin(
    bundle: Trajectories,
    bin_name: str,
    *,
    seed: int | None = None,
) -> int:
    """Draw one trajectory index uniformly from the given encounter bin."""
    if bundle.encounter_bins is None or bundle.encounter_bin_id is None:
        msg = "test bundle is not stratified"
        raise ValueError(msg)
    bin_names = [b.name for b in bundle.encounter_bins]
    if bin_name not in bin_names:
        msg = f"unknown bin {bin_name!r}; expected one of {bin_names}"
        raise ValueError(msg)
    bin_id = bin_names.index(bin_name)
    candidates = np.flatnonzero(bundle.encounter_bin_id == bin_id)
    if candidates.size == 0:
        msg = f"bin {bin_name!r} has no trajectories"
        raise ValueError(msg)
    rng = np.random.default_rng(seed)
    return int(rng.choice(candidates))


def mean_rollout_position_mse(
    true_traj: np.ndarray,
    predicted: np.ndarray,
) -> float:
    """Rollout-averaged position MSE for one trajectory (audience-facing forecast metric)."""
    diff = predicted[..., :2] - true_traj[..., :2]
    return float((diff**2).mean())


def plot_trajectory_quad(
    true_traj: np.ndarray,
    egnn_pred: np.ndarray,
    hgnn_pred: np.ndarray,
    baseline_pred: np.ndarray,
    *,
    trajectory_index: int,
    bin_name: str,
    d_min: float,
) -> None:
    """Render one row of four panels comparing all three predictors against truth."""
    panels = [
        ("ground truth", true_traj, None),
        ("EGNN", egnn_pred, mean_rollout_position_mse(true_traj, egnn_pred)),
        ("HGNN", hgnn_pred, mean_rollout_position_mse(true_traj, hgnn_pred)),
        ("constant velocity", baseline_pred, mean_rollout_position_mse(true_traj, baseline_pred)),
    ]
    x_lim, y_lim = _shared_quad_limits([p[1] for p in panels])
    n_particles = true_traj.shape[1]
    fig, axes = plt.subplots(1, 4, figsize=(20, 5.2), sharex=True, sharey=True)
    fig.suptitle(
        f"trajectory {trajectory_index} | bin={bin_name} | d_min={d_min:.4g}",
        fontsize=14,
    )
    for ax, (title, data, mse) in zip(axes, panels, strict=True):
        ax.set_xlim(*x_lim)
        ax.set_ylim(*y_lim)
        ax.set_aspect("equal")
        ax.grid(True, alpha=0.3)
        ax.set_title(title if mse is None else f"{title} (mean position MSE={mse:.4g})")
        for p in range(n_particles):
            color = _particle_color(p)
            ax.plot(data[:, p, 0], data[:, p, 1], color=color, alpha=0.7, linewidth=1.0)
            ax.scatter(data[0, p, 0], data[0, p, 1], color=color, marker="o", s=30, zorder=5)
            ax.scatter(data[-1, p, 0], data[-1, p, 1], color=color, marker="x", s=40, zorder=5)
    fig.tight_layout(rect=(0.0, 0.0, 1.0, 0.94))
    plt.show()


def animate_trajectory_quad(
    true_traj: np.ndarray,
    egnn_pred: np.ndarray,
    hgnn_pred: np.ndarray,
    baseline_pred: np.ndarray,
    *,
    trajectory_index: int,
    bin_name: str,
    d_min: float,
    fps: int = 20,
) -> None:
    """Render an inline four-panel animation for the current trajectory."""
    if fps < 1:
        msg = f"fps must be >= 1; got {fps}"
        raise ValueError(msg)

    from IPython.display import HTML, display
    from matplotlib.animation import FuncAnimation

    panels = [
        ("ground truth", true_traj),
        ("EGNN", egnn_pred),
        ("HGNN", hgnn_pred),
        ("constant velocity", baseline_pred),
    ]
    x_lim, y_lim = _shared_quad_limits([p[1] for p in panels])
    n_frames, n_particles, _ = true_traj.shape

    fig, axes = plt.subplots(1, 4, figsize=(20, 5.2), sharex=True, sharey=True)
    fig.suptitle(
        f"trajectory {trajectory_index} | bin={bin_name} | d_min={d_min:.4g}",
        fontsize=14,
    )

    trails: list[list] = []
    dots: list[list] = []
    for ax, (title, _data) in zip(axes, panels, strict=True):
        ax.set_xlim(*x_lim)
        ax.set_ylim(*y_lim)
        ax.set_aspect("equal")
        ax.grid(True, alpha=0.3)
        ax.set_title(title)
        panel_trails = []
        panel_dots = []
        for p in range(n_particles):
            color = _particle_color(p)
            panel_trails.append(ax.plot([], [], color=color, alpha=0.45, linewidth=1.0)[0])
            panel_dots.append(ax.plot([], [], "o", color=color, markersize=7)[0])
        trails.append(panel_trails)
        dots.append(panel_dots)

    artists = [artist for panel in trails + dots for artist in panel]

    def _init() -> list:
        for artist in artists:
            artist.set_data([], [])
        return artists

    def _update(frame: int) -> list:
        for panel_index, (_title, data) in enumerate(panels):
            for p in range(n_particles):
                trails[panel_index][p].set_data(data[: frame + 1, p, 0], data[: frame + 1, p, 1])
                dots[panel_index][p].set_data([data[frame, p, 0]], [data[frame, p, 1]])
        return artists

    anim = FuncAnimation(
        fig,
        _update,
        init_func=_init,
        frames=n_frames,
        interval=max(1, 1000 // fps),
        blit=True,
    )
    fig.tight_layout(rect=(0.0, 0.0, 1.0, 0.94))
    plt.close(fig)
    display(HTML(anim.to_jshtml()))


def _shared_quad_limits(
    panels: list[np.ndarray],
) -> tuple[tuple[float, float], tuple[float, float]]:
    """Compute one (xlim, ylim) pair that fits every panel's finite positions."""
    xs: list[float] = []
    ys: list[float] = []
    for panel in panels:
        finite = np.isfinite(panel[..., 0]) & np.isfinite(panel[..., 1])
        if not finite.any():
            continue
        xs.extend([float(panel[..., 0][finite].min()), float(panel[..., 0][finite].max())])
        ys.extend([float(panel[..., 1][finite].min()), float(panel[..., 1][finite].max())])
    if not xs or not ys:
        return (-1.0, 1.0), (-1.0, 1.0)
    pad = 0.5
    return (min(xs) - pad, max(xs) + pad), (min(ys) - pad, max(ys) + pad)

## Pick a random trajectory from `BIN_NAME`

Rerun this cell as many times as you want; each rerun samples a fresh index
(unless `RANDOM_SEED` is pinned).

In [ ]:
idx = random_trajectory_from_bin(test_bundle, BIN_NAME, seed=RANDOM_SEED)
d_min = float(test_bundle.min_pairwise_distance[idx])
print(f"bin={BIN_NAME} | trajectory={idx} | d_min={d_min:.4g}")

## Preview the four panels for the current `idx`

In [ ]:
plot_trajectory_quad(
    test_bundle.states[idx],
    egnn_predicted[idx],
    hgnn_predicted[idx],
    baseline_predicted[idx],
    trajectory_index=idx,
    bin_name=BIN_NAME,
    d_min=d_min,
)

## Animate the current `idx`

Optional inline animation for the trajectory you just previewed. This reuses the
already-computed rollouts, so it does not rerun evaluation.


In [ ]:
ANIMATE_CURRENT = True
ANIMATION_FPS = 20

if ANIMATE_CURRENT:
    animate_trajectory_quad(
        test_bundle.states[idx],
        egnn_predicted[idx],
        hgnn_predicted[idx],
        baseline_predicted[idx],
        trajectory_index=idx,
        bin_name=BIN_NAME,
        d_min=d_min,
        fps=ANIMATION_FPS,
    )

## Cherry-pick

Edit the `SELECTED` mapping below as you find indices you want to keep. The
notebook does not enforce that you commit anything before saving; the final
cell validates the mapping you wrote here.

In [ ]:
SELECTED = {
    "close": 0,
    "near": 0,
    "mid": 0,
    "wide": 0,
    "far": 0,
}

## Save `SELECTED` to `SELECTION_PATH`

Writes a YAML file in the exact format consumed by
`python -m evaluation.animate_best --selection-file <path>`. Validation is strict:

- Every test-set bin must appear exactly once and only known bins are allowed.
- Each index must be a non-negative integer within the trajectory range.
- Each index must actually live in the bin assigned in `SELECTED`.

In [ ]:
def write_selection_yaml(
    selected: dict[str, int],
    path: Path,
    bundle: Trajectories,
    egnn_pred: np.ndarray,
    hgnn_pred: np.ndarray,
    baseline_pred: np.ndarray,
) -> None:
    """Validate `selected` against the test bundle and persist it as YAML."""
    if bundle.encounter_bins is None or bundle.encounter_bin_id is None:
        msg = "test bundle is not stratified"
        raise ValueError(msg)
    bin_names = [b.name for b in bundle.encounter_bins]
    missing = sorted(set(bin_names) - set(selected.keys()))
    unknown = sorted(set(selected.keys()) - set(bin_names))
    if missing:
        msg = f"SELECTED is missing bins: {missing}. Expected one entry per encounter bin: {bin_names}"
        raise ValueError(msg)
    if unknown:
        msg = f"SELECTED has unknown bin names: {unknown}. Expected one entry per encounter bin: {bin_names}"
        raise ValueError(msg)
    n_traj = bundle.states.shape[0]
    ordered: dict[str, int] = {}
    for bin_id, bin_def in enumerate(bundle.encounter_bins):
        value = selected[bin_def.name]
        if isinstance(value, bool) or not isinstance(value, int):
            msg = f"index for bin {bin_def.name!r} must be an integer; got {type(value).__name__}"
            raise ValueError(msg)
        if value < 0 or value >= n_traj:
            msg = (
                f"index {value} for bin {bin_def.name!r} is out of range (n_trajectories={n_traj})"
            )
            raise ValueError(msg)
        actual_bin_id = int(bundle.encounter_bin_id[value])
        if actual_bin_id != bin_id:
            actual_name = bin_names[actual_bin_id]
            msg = (
                f"trajectory {value} is assigned to bin {bin_def.name!r} but lives in bin "
                f"{actual_name!r} in the test set"
            )
            raise ValueError(msg)
        _require_finite_selection_prediction("EGNN", bin_def.name, value, bundle.states, egnn_pred)
        _require_finite_selection_prediction("HGNN", bin_def.name, value, bundle.states, hgnn_pred)
        _require_finite_selection_prediction(
            "constant velocity", bin_def.name, value, bundle.states, baseline_pred
        )
        ordered[bin_def.name] = value
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w") as f:
        yaml.safe_dump(ordered, f, sort_keys=False)


def _require_finite_selection_prediction(
    model_name: str,
    bin_name: str,
    idx: int,
    states: np.ndarray,
    predicted: np.ndarray,
) -> None:
    """Reject selected examples whose displayed prediction has non-finite MSE."""
    mse = mean_rollout_position_mse(states[idx], predicted[idx])
    if not np.isfinite(mse):
        msg = f"{model_name} prediction for selected {bin_name!r} trajectory {idx} is non-finite"
        raise ValueError(msg)


write_selection_yaml(
    SELECTED,
    SELECTION_PATH,
    test_bundle,
    egnn_predicted,
    hgnn_predicted,
    baseline_predicted,
)
print(f"wrote selection to {SELECTION_PATH}")
print(
    f"run animations with: python -m evaluation.animate_best --selection-file {SELECTION_PATH} ..."
)